# Optimization

examples

In [57]:
import numpy as np
from scipy import linalg as la
import scipy.optimize as opt
import matplotlib.pyplot as plt
from numpy.polynomial import Polynomial
np.set_printoptions(suppress=True, precision=6)  # 抑制科学计数法，小数位6位

def sign(x):
    s = np.sign(x)
    return s if s else 1

In [58]:
# 6.6
def f(x):
    return np.array([
        2 * x[0] + x[1] + x[0] * x[1] * x[2],
        2 * x[0] + x[0] ** 2 * x[2],
        x[0] ** 2 * x[1] - 1000 / np.pi, 
    ])

opt.root(f, x0=[1, 2, 3]).x

array([ 5.419261, 10.838521, -0.369054])

In [59]:
B = np.array([
    [-12.6, -6.3],
    [-6.3, 0],
])
Q, R = la.qr(np.array([369, 92.3]).reshape(-1, 1))
Q[:, 1] @ B @ Q[:, 1]

np.float64(2.2241907726339227)

In [60]:
# 6.8
def golden_section_search(f, a, b, n=1000, tol=1e-6):
    r = (np.sqrt(5) - 1) / 2
    x1, x2 = a + (1 - r) * (b - a), a + r * (b - a)
    fx1, fx2 = f(x1), f(x2)
    history = [[x1, fx1, x2, fx2]]
    for _ in range(n):
        if fx1 < fx2:
            b = x2
            x2, fx2 = x1, fx1
            x1 = a + (1 - r) * (b - a)
            fx1 = f(x1)
        else:
            a = x1
            x1, fx1 = x2, fx2
            x2 = a + r * (b - a)
            fx2 = f(x2)
        history.append([x1, fx1, x2, fx2])
        if b - a < tol:
            break
    
    return np.array(history)


def f(x):
    return .5 - x * np.exp(- x ** 2)


golden_section_search(f, 0, 2, n=13)
    

array([[0.763932, 0.073809, 1.236068, 0.231775],
       [0.472136, 0.122204, 0.763932, 0.073809],
       [0.763932, 0.073809, 0.944272, 0.112868],
       [0.652476, 0.07374 , 0.763932, 0.073809],
       [0.583592, 0.084857, 0.652476, 0.07374 ],
       [0.652476, 0.07374 , 0.695048, 0.071243],
       [0.695048, 0.071243, 0.72136 , 0.071291],
       [0.678787, 0.071815, 0.695048, 0.071243],
       [0.695048, 0.071243, 0.705098, 0.071122],
       [0.705098, 0.071122, 0.71131 , 0.071133],
       [0.70126 , 0.071147, 0.705098, 0.071122],
       [0.705098, 0.071122, 0.707471, 0.071118],
       [0.707471, 0.071118, 0.708937, 0.071121],
       [0.706565, 0.071118, 0.707471, 0.071118]])

In [61]:
# 6.9
def successive_parabolic_interpolation(f, u, v, w, n=1000, tol=1e-6):
    history = []
    fu, fv, fw = f(u), f(v), f(w)
    if fu < fv:
        u, fu, v, fv = v, fv, u, fu
    if fw < fv:
        w, fw, v, fv = v, fv, w, fw

    for _ in range(n):
        p = (v - u) ** 2 * (fv - fw) - (v - w) ** 2 * (fv - fu)
        q = 2 * ((v - u) * (fv - fw) - (v - w) * (fv - fu))
        if abs(q) < tol:
            break
        h = - p / q
        u, w, v = w, v, v + h
        fu, fv, fw = fw, f(v), fv
        history.append([v, fv])
        if abs(h) < tol:
            break
    return v, np.array(history)

# successive_parabolic_interpolation(f, 0, .6, 1.2, n=5)
# print(f(1))

_, history = successive_parabolic_interpolation(f, 0, .6, 1.2, n=5)
print(history)

[[0.754267 0.072981]
 [0.720797 0.071278]
 [0.708374 0.071119]
 [0.70692  0.071118]]


In [62]:
# 6.10
def fprime(x):
    return (2 * x ** 2 - 1) * np.exp(- x ** 2)


def fdoubleprime(x):
    return 2 * x * (3 - 2 * x ** 2) * np.exp(- x ** 2)


def newton(f, fprime, x, tol=1e-6, n=1000):
    history = []
    for _ in range(n):
        fx, fprimex = f(x), fprime(x)
        h = - fx / fprimex
        history.append([x, fx, fprimex, h])
        if abs(h) < tol:
            break
        x += h
    return x, np.array(history)


_, history = newton(fprime, fdoubleprime, 1)
print(history)

[[ 1.        0.367879  0.735759 -0.5     ]
 [ 0.5      -0.3894    1.947002  0.2     ]
 [ 0.7      -0.012253  1.732507  0.007072]
 [ 0.707072 -0.000059  1.715612  0.000035]
 [ 0.707107 -0.        1.715528  0.      ]]


In [68]:
# 6.11
def f(x):
    return .5 * x[0] ** 2 + 2.5 * x[1] ** 2


def fprime(x):
    return np.array([x[0], 5 * x[1]])


def steepest_descent(f, fprime, x, maxstep=1, n=1000, tol=1e-4):
    history = []
    x = np.array(x, dtype=float)
    for _ in range(n):
        s = -fprime(x)
        h, _ = successive_parabolic_interpolation(lambda a: f(x + a * s), u=0, v=maxstep/2, w=maxstep, tol=tol)
        history.append(np.hstack([x, [f(x)], fprime(x), [la.norm(h * s)]]))
        if la.norm(h * s)< tol:
            break
        x += h * s
    
    return x, np.array(history)


_, history = steepest_descent(f, fprime, [5, 1], n=10)
print(history.round(3))


[[ 5.     1.    15.     5.     5.     2.357]
 [ 3.333 -0.667  6.667  3.333 -3.333  1.571]
 [ 2.222  0.444  2.963  2.222  2.222  1.048]
 [ 1.481 -0.296  1.317  1.481 -1.481  0.698]
 [ 0.988  0.198  0.585  0.988  0.988  0.466]
 [ 0.658 -0.132  0.26   0.658 -0.658  0.31 ]
 [ 0.439  0.088  0.116  0.439  0.439  0.207]
 [ 0.293 -0.059  0.051  0.293 -0.293  0.138]
 [ 0.195  0.039  0.023  0.195  0.195  0.092]
 [ 0.13  -0.026  0.01   0.13  -0.13   0.061]]


In [71]:
# 6.12
def n_newton(f, jac, x, args=(), n=1000, tol=1e-4):
    history = []
    for _ in range(n):
        h = la.solve(jac(x, *args), -f(x, *args))
        history.append(np.hstack([x, [la.norm(h, ord=np.inf)]]))
        if la.norm(h) < tol:
            break
        x += h
    return x, np.array(history)


def hessian(x):
    return np.array([
        [1, 0],
        [0, 5],
    ])


_, history = n_newton(fprime, hessian, x=[5, 1])
print(history)

[[5. 1. 5.]
 [0. 0. 0.]]


In [75]:
# 6.13
def bfgs(f, x, n=1000, tol=1e-4):
    x = np.array(x, dtype=float)
    B = np.eye(len(x))
    fx0 = f(x)
    history = []
    for _ in range(n):
        s = la.solve(B, -fx0)
        x += s
        fx = f(x)
        history.append(np.hstack([x, fx, [la.norm(fx)]]))
        if la.norm(fx) < tol:
            break
        y = fx - fx0
        fx0 = fx
        B += np.outer(y, y) / (y @ s) - (B @ np.outer(s, s) @ B) / (s @ B @ s)
    
    return x, np.array(history)


_, history = bfgs(fprime, [5, 1])
print(history.round(3))

[[  0.     -4.      0.    -20.     20.   ]
 [ -2.222   0.444  -2.222   2.222   3.143]
 [  0.816   0.082   0.816   0.408   0.913]
 [ -0.009  -0.015  -0.009  -0.077   0.077]
 [ -0.001   0.001  -0.001   0.005   0.005]
 [  0.     -0.      0.     -0.      0.   ]]


In [ ]:
def conjugate_gradient(f, fprime, x, maxstep=1, tol=1e-4):
    history = []
    x = np.array(x, dtype=float)
    g0 = fprime(x)
    s = - g0
    for _ in range(len(x)):
        h, _ = successive_parabolic_interpolation(lambda a: f(x + a * s), u=0, v=maxstep/2, w=maxstep, tol=tol)
        if la.norm(h * s) < tol:
            break
        x += h * s
        g = fprime(x)
        b = (g @ g) / (g0 @ g0)
        s = - g + b * s
        g0 = g
        history.append(np.hstack([x, s, [la.norm(h * s)]]))
    
    return x, np.array(history)


_, history = conjugate_gradient(f, fprime, [5, 1])
print(history.round(3))

0.3333333333333333
0.6000000000000001
[[ 3.333 -0.667 -5.556  1.111  1.889]
 [-0.     0.     0.     0.     0.   ]]
